<b style="color:red;"> Research question: </b> <b>How present is bias (across sensitive attributes) and fairness in the generated datasets? </b>

In [1]:
# Imports 
import pandas as pd
import re
import numpy as np
import os
from pathlib import Path
import glob
from scipy import stats
from collections import defaultdict
from scipy.spatial.distance import jensenshannon
from sklearn.metrics import mutual_info_score
from scipy.stats import chi2_contingency, ks_2samp, wasserstein_distance
import warnings
warnings.filterwarnings('ignore')


In [2]:
from functions import find_project_root

# Finding the project root
PROJECT_ROOT = find_project_root()

# Directory to save the generated datasets 
GENERATED_DIR = PROJECT_ROOT / "data" / "generated"

<b style="color:yellow;"> In this cell we load the datasets for each domain. There is approx. 27 datasets per each domain. That is because we have 3 shots (zero, one and few), 3 LLMs and 3 runs for each. </b>

In [3]:
from functions import list_generated_datasets

# Getting the metadata table. This table contains information about the domain, model, shot, run and file path.
metadata_table = list_generated_datasets(GENERATED_DIR)
metadata_table

Found 81 CSV files.


,domain,model,shot,run,file_path
0,employment,kimi-k2-instruct-0905,few,run2,/Users/estref/Desktop/master_thesis/data/gener...
1,employment,kimi-k2-instruct-0905,one,run2,/Users/estref/Desktop/master_thesis/data/gener...
2,employment,kimi-k2-instruct-0905,zero,run2,/Users/estref/Desktop/master_thesis/data/gener...
3,lending,kimi-k2-instruct-0905,few,run2,/Users/estref/Desktop/master_thesis/data/gener...
4,lending,kimi-k2-instruct-0905,one,run2,/Users/estref/Desktop/master_thesis/data/gener...
...,...,...,...,...,...
76,lending,qwen3-coder-30b-a3b-instruct,one,run3,/Users/estref/Desktop/master_thesis/data/gener...
77,lending,qwen3-coder-30b-a3b-instruct,zero,run3,/Users/estref/Desktop/master_thesis/data/gener...
78,hatecrime,qwen3-coder-30b-a3b-instruct,few,run3,/Users/estref/Desktop/master_thesis/data/gener...
79,hatecrime,qwen3-coder-30b-a3b-instruct,one,run3,/Users/estref/Desktop/master_thesis/data/gener...


In [4]:
from functions import load_csv_safely

# Create empty column first where the clean data will be stored
metadata_table["data"] = None   


#In this step we load the data from the csv files, skip bad lines and keep the data clean in order to work with it
for i in range(len(metadata_table)):
    file_path = metadata_table.loc[i, "file_path"]
    df = load_csv_safely(file_path)
    metadata_table.at[i, "data"] = df


# Sample of getting a dataset
# metadata_table.iloc[5].data

<b style="color:yellow;"> Now it is time to load the real datasets for each domain. </b>

In [5]:
REAL_DATA_DIR = PROJECT_ROOT / "data" / "preprocessed"

#Load employment data
employment_real = pd.read_csv(REAL_DATA_DIR / "uk_gender_pay_gap_data_2024_to_2025_preproccesed.csv")


# Load lending data
lending_real = pd.read_csv(REAL_DATA_DIR / "year_2024_preprocessed.csv")


# Load hate crime data
hatecrime_real = pd.read_csv(REAL_DATA_DIR / "hate_crime_preprocessed.csv")

print("Load the real employment dataset: ", employment_real.shape)
print("Load the real lending dataset: ", lending_real.shape)
print("Load the real hate crime dataset: ", hatecrime_real.shape)


Load the real employment dataset:  (11239, 18)
Load the real lending dataset:  (12229298, 58)
Load the real hate crime dataset:  (265834, 20)


<b style="color:yellow;"> As it is described in the thesis, sensitive and outcome attributes are defined for each of our domains. </b>

First, we define the sensitive atttributes, then if the sensitive attributes are of type string, the unique values for each sensitive attribute is obtained for the Real and LLM generated datasets, and we map the latter so they match the ones of the real dataset. The mapping are saved as a new column, called {original_column_name}_mapped}

In [6]:
#Define the sensitive attributes for each domain

SENSITIVE_ATTRIBUTES = {
    
    # Employment dataset has no sensitive attributes whose values are strings, but I am still providing the code for future reference
    # "employment": ["FemaleBonusPercent", "MaleBonusPercent", "FemaleLowerQuartile", "MaleLowerQuartile", 
    #                "FemaleLowerMiddleQuartile", "MaleLowerMiddleQuartile", "FemaleUpperMiddleQuartile", 
    #                "MaleUpperMiddleQuartile", "FemaleTopQuartile", "MaleTopQuartile"],

    "lending": ["derived_ethnicity", "derived_race", "derived_sex"],

    "hatecrime": ["offender_race", "offender_ethnicity"] 
}


OUTCOME_ATTRIBUTES = {
    # Currently none of the tables have this outcome attribute but it will be created.
    "lending": ["action_taken"],
    "hatecrime": ["offense_name"]
}


In [7]:
# Get the unique values for each sensitive attribute, for the real and LLM generated dataset
from functions import print_unique_sensitive_values

# Combine all LLM-generated datasets by domain
llm_data_by_domain = {
    domain: pd.concat(
        [row["data"] for _, row in metadata_table[metadata_table["domain"] == domain].iterrows()],
        ignore_index=True
    )
    for domain in ["lending", "hatecrime"] # Employment dataset has no sensitive attributes whose values are strings
}

real_data_by_domain = {
   # "employment": employment_real, Commented out because the employment dataset has no sensitive attributes whose values are strings
    "lending": lending_real,
    "hatecrime": hatecrime_real
}

print_unique_sensitive_values(SENSITIVE_ATTRIBUTES, real_data_by_domain, llm_data_by_domain)


DOMAIN: LENDING

Sensitive Attribute: derived_ethnicity
  • Real unique values (5):
      ['Ethnicity Not Available', 'Free Form Text Only', 'Hispanic or Latino', 'Joint', 'Not Hispanic or Latino']
  • LLM unique values (119):
      [' "Hispanic or Latino"', ' "Not Hispanic or Latino"', '1', '1-4 Family Home', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '3', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '4', '40', '5', '6', '7', '8', '9', 'African American', 'Alaskan Native', 'American Indian', 'American Indian or Alaska Native', 'Application denied', 'Asian', 'Asian or Pacific Islander', 'Black', 'Black or African American', 'Black or African Female', 'Black/African American', 'Brown', 'Central American', 'Ethnicity Not Available', 'European American/White', 'Female', 'File closed for incompleteness', 'HISPIDIAN OR LATINO', 'Hawaiian/Pacific Islander', 'His', 'Hisp or Latino', 'Hisp. or Lati

<b style="color:yellow;"> Mapping the values of the LLM (where it is possible), to the values of the Real Dataset. If it is not possible I am marking it as NaN in the new column. </b>

In [8]:
# Dictionary used for the mapping of the LLM generated datasets to the real datasets, it can be 
# found in the functions.py file. I have manually mapped all the values that were printed above.
from functions import ATTRIBUTE_ALIASES

In [9]:
from functions import apply_mappings_to_all_datasets

# Function that applied the mappings to the LLM generated dataset, so that it can be compared to the real dataset.
apply_mappings_to_all_datasets(metadata_table, ATTRIBUTE_ALIASES)

# Checking if the mappings was succesfull


# test_df = metadata_table.iloc[33].data

# print(test_df["offender_race_Mapped"].dropna().unique())
# print(test_df["offender_ethnicity_Mapped"].dropna().unique())

# print(test_df["derived_ethnicity_Mapped"].dropna().unique())
# print(test_df["derived_race_Mapped"].dropna().unique())
# print(test_df["derived_sex_Mapped"].dropna().unique())




In [10]:
from functions import find_bad_mappings_for_config


# This function is used to find the 'bad' mappings, that means mappings with a high rate of missing values.
# The threshold is set to 5% of the total number of mappings, in this case. But it can be increased or decreased according 
# to need.

bad_mappings_zero = find_bad_mappings_for_config(
    metadata_table,
    SENSITIVE_ATTRIBUTES,
    domain="hatecrime",
    model="kimi-k2-instruct-0905",
    #model="llama-3.1-8b-instruct",
    #model="qwen3-coder-30b-a3b-instruct",
    run="run3",
    shot="zero",
    threshold=0.05, 
)


No mappings above 5% NaN for this config.


<b style="color:yellow;"> Similar to the sensitive attributes, we are defining the outcome attributes and crreating a new column, in order to have a binary output. </b>

In [11]:
from functions import print_unique_outcome_values

# In this same line of work I print the unique values for real and LLM generated datasets for the outcome attributes
print_unique_outcome_values(
    OUTCOME_ATTRIBUTES,
    real_data_by_domain,
    llm_data_by_domain
)


DOMAIN: LENDING

Outcome Attribute: action_taken
  • Real unique values (11):
      ['File closed for incompleteness', 'Loan originated', 'Application withdrawn', 'Application denied', 'Application approved but not accepted', '6', '7', '8', 8, 7, 6]
  • LLM unique values (429):
      ['Loan originated', 'Application denied', 'File closed for incompleteness', 'Loan purchased by the lender', 'Application approved but not accepted', 'Application withdrawn', '1', 'action_taken', '3', '2', 'Preapproval request denied', '4', '5', '6', 'Male', 'Female', 'Loan purchased by financial institution', 'Preapproval request approved but not accepted', '7', '8', '32', '42', 'Home purchase', 'Refinance', 'Home improvement', '31', 'Refinancing', 'Cash-out refinancing', 'Other', 'Loan submitted but not accepted for underwriting', 'Approved loan not submitted for purchase', 'loan_purpose', '33', '34', 'Application accepted', 'Loan closed', 'Loan approved but not accepted', 'Loan denied', 'Loan approved bu

In [12]:
from functions import OUTCOME_ALIASES

In [13]:
from functions import apply_outcome_mappings_all
apply_outcome_mappings_all(real_data_by_domain, metadata_table, OUTCOME_ALIASES)


Outcome columns created on all REAL and LLM datasets successfully.


In [14]:
# Testing if the mappings worked
subset = metadata_table[
    (metadata_table["domain"] == "lending")
].iloc[0]
df_llm = subset["data"]
df_llm["loan_approved"].head()



0    1.0
1    1.0
2    0.0
3    1.0
4    1.0
Name: loan_approved, dtype: float64

In [15]:
# Testing

subset = metadata_table[
    (metadata_table["domain"] == "hatecrime")
].iloc[0]
df_hc = subset["data"]
df_hc["is_violent"].head()

0    non-violent
1    non-violent
2        violent
3        violent
4    non-violent
Name: is_violent, dtype: object

In [16]:
# Testing

subset = metadata_table[
    (metadata_table["domain"] == "lending") &
    (metadata_table["model"] == "kimi-k2-instruct-0905") &
    (metadata_table["run"]   == "run1") &
    (metadata_table["shot"]  == "few")
]

df = subset.iloc[0]["data"]

df["loan_approved"].value_counts(dropna=False)


loan_approved
1.0    728
0.0    102
NaN      4
Name: count, dtype: int64

<b style="color:yellow;"> After defining the senstive and outcome attributes, now we calulcate the Metrics. Starting with the domain of <b style="color:red;"> EMPLOYMENT</b>

<b style="color:yellow;"> 1. Base Rate Parity </b>

In [17]:
# Function to calculate the base rate parity, this is for the Employment dataset, in this occasion  only
# the (Male/Female) pairs are needed. 
from functions import base_rate_parity_employment

EMPLOYMENT_PAIRS = [("MaleBonusPercent", "FemaleBonusPercent"), 
                    ("MaleLowerQuartile", "FemaleLowerQuartile"), 
                    ("MaleLowerMiddleQuartile", "FemaleLowerMiddleQuartile"), 
                    ("MaleUpperMiddleQuartile", "FemaleUpperMiddleQuartile"), 
                    ("MaleTopQuartile", "FemaleTopQuartile")]


# TEST
# print(" The average base rate parity for the employment dataset is:", base_rate_parity_employment(employment_real, EMPLOYMENT_PAIRS)[0])
 
# Looping through all the LLM generated datasets and calculating the base rate parity for the Employment dataset

results = []

for idx, row in metadata_table.iterrows():
    if row["domain"] != "employment":
        continue
    
    df = row["data"]        # the actual dataset
    model = row["model"]
    run   = row["run"]
    shot  = row["shot"]

    bp_avg, bp_values = base_rate_parity_employment(df, EMPLOYMENT_PAIRS)

    results.append({
        "model": model,
        "run": run,
        "shot": shot,
        "bp_avg": bp_avg,
       #"bp_per_pair": bp_values,
    })

# Convert to DF for easy viewing
employment_bp_results = pd.DataFrame(results)

# Save the results in a csv file
employment_bp_results.to_csv("employment_e01_base_rate_parity.csv", index=False)
output_path = PROJECT_ROOT / "analysis" / "bias_rq_02" / "employment_e01_base_rate_parity.csv"
employment_bp_results.to_csv(output_path, index=False)
print("Saved to:", output_path)


Saved to: /Users/estref/Desktop/master_thesis/analysis/bias_rq_02/employment_e01_base_rate_parity.csv


<b style="color:yellow;"> 2. Disparate Impact </b>

In [18]:
from functions import disparate_impact_employment

results = []

for idx, row in metadata_table.iterrows():
    if row["domain"] != "employment":
        continue

    df    = row["data"]
    model = row["model"]
    run   = row["run"]
    shot  = row["shot"]

    di_avg, di_values = disparate_impact_employment(df, EMPLOYMENT_PAIRS)

    results.append({
        "model": model,
        "run": run,
        "shot": shot,
        "di_avg": di_avg,
        # "di_per_pair": di_values,   # optional
    })

employment_di_results = pd.DataFrame(results)

# Testing for real data
# print(disparate_impact_employment(employment_real, EMPLOYMENT_PAIRS)[0])

# Save the results in a csv file
employment_di_results.to_csv("employment_e02_disparate_impact.csv", index=False)
output_path = PROJECT_ROOT / "analysis" / "bias_rq_02" / "employment_e02_disparate_impact.csv"
employment_di_results.to_csv(output_path, index=False)
print("Saved to:", output_path)


Saved to: /Users/estref/Desktop/master_thesis/analysis/bias_rq_02/employment_e02_disparate_impact.csv


<b style="color:yellow;"> 3. Mean Difference </b>

In [19]:
from functions import mean_difference_employment

results_md = []

for idx, row in metadata_table.iterrows():
    if row["domain"] != "employment":
        continue

    df    = row["data"]
    model = row["model"]
    run   = row["run"]
    shot  = row["shot"]

    md_avg, md_values = mean_difference_employment(df, EMPLOYMENT_PAIRS)

    results_md.append({
        "model": model,
        "run": run,
        "shot": shot,
        "md_avg": md_avg,
        # "md_per_pair": md_values,   # keep commented if you only want one column
    })

employment_md_results = pd.DataFrame(results_md)
#print(employment_md_results)

# Real data testing
#print(mean_difference_employment(employment_real, EMPLOYMENT_PAIRS)[0])

# Save the results in a csv file
employment_md_results.to_csv("employment_e03_mean_difference.csv", index=False)
output_path = PROJECT_ROOT / "analysis" / "bias_rq_02" / "employment_e03_mean_difference.csv"
employment_md_results.to_csv(output_path, index=False)
print("Saved to:", output_path)




Saved to: /Users/estref/Desktop/master_thesis/analysis/bias_rq_02/employment_e03_mean_difference.csv


In [20]:
# Put into a DataFrame
real_metrics_df = pd.DataFrame([{
    "bp_avg": base_rate_parity_employment(employment_real, EMPLOYMENT_PAIRS)[0],
    "di_avg": disparate_impact_employment(employment_real, EMPLOYMENT_PAIRS)[0],
    "md_avg": mean_difference_employment(employment_real, EMPLOYMENT_PAIRS)[0]
}])

# Save to same directory you used earlier
output_path = PROJECT_ROOT / "analysis" / "bias_rq_02" / "employment_real_metrics.csv"
real_metrics_df.to_csv(output_path, index=False)
print("Saved real metrics to:", output_path)

Saved real metrics to: /Users/estref/Desktop/master_thesis/analysis/bias_rq_02/employment_real_metrics.csv


<b style="color:yellow;"> Calculating the metrics for <b style="color:red;"> HATECRIME</b>

<b style="color:yellow;"> 1. Base Rate Parity</b>

In [30]:
from functions import base_rate_parity


results = []

for idx, row in metadata_table.iterrows():
    if row["domain"] != "hatecrime":
        continue

    df = row["data"]        # The actual LLM-generated dataset
    model = row["model"]
    run   = row["run"]
    shot  = row["shot"]

    overall_bp, per_attr_bp, per_attr_probs = base_rate_parity(
        df,
        sensitive_attrs=["offender_ethnicity_Mapped", "offender_race_Mapped"],
        outcome_col="is_violent",
        positive_label="non-violent"
    )

    results.append({
        "model": model,
        "run": run,
        "shot": shot,
        "brp_overall": overall_bp,
        "brp_ethnicity": per_attr_bp.get("offender_ethnicity_Mapped"),
        "brp_race": per_attr_bp.get("offender_race_Mapped"),
    })

hatecrime_brp_results = pd.DataFrame(results)
#print(hatecrime_brp_results)


# Save the results in a csv file
output_path = PROJECT_ROOT / "analysis" / "bias_rq_02" / "hatecrime_e01_base_rate_parity.csv"
hatecrime_brp_results.to_csv(output_path, index=False)
print("Saved real metrics to:", output_path)


Saved real metrics to: /Users/estref/Desktop/master_thesis/analysis/bias_rq_02/hatecrime_e01_base_rate_parity.csv


<b style="color:yellow;"> 2. Disparate Impact </b>

In [32]:
from functions import disparate_impact_multiclass

results = []

for idx, row in metadata_table.iterrows():
    if row["domain"] != "hatecrime":
        continue

    df = row["data"]        # The actual LLM-generated dataset
    model = row["model"]
    run   = row["run"]
    shot  = row["shot"]

    overall_di, per_attr_di, per_attr_di_probs = disparate_impact_multiclass(
        df,
        sensitive_attrs=["offender_ethnicity_Mapped", "offender_race_Mapped"],
        outcome_col="is_violent",
        positive_label="non-violent"
    )

    results.append({
        "model": model,
        "run": run,
        "shot": shot,
        "di_overall": overall_di,
        "di_race": per_attr_di.get("offender_race_Mapped"),
        "di_ethnicity": per_attr_di.get("offender_ethnicity_Mapped"),
    })

hatecrime_di_results = pd.DataFrame(results)
#print(hatecrime_di_results)

#Save the results in a csv file
output_path = PROJECT_ROOT / "analysis" / "bias_rq_02" / "hatecrime_e02_disparate_impact.csv"
hatecrime_di_results.to_csv(output_path, index=False)
print("Saved real metrics to:", output_path)

Saved real metrics to: /Users/estref/Desktop/master_thesis/analysis/bias_rq_02/hatecrime_e02_disparate_impact.csv


<b style="color:yellow;"> 3. BASE RATE </b>

In [34]:
from functions import base_rate_multiclass
# br_overall_real, br_per_attr_real, br_rates_real = base_rate_multiclass(
#     hatecrime_real,
#     sensitive_attrs=HATECRIME_SENSITIVE,
#     outcome_col=HATECRIME_OUTCOME_COL,
#     positive_label=HATECRIME_POSITIVE_LABEL,
# )

# br_overall_real

results = []

for idx, row in metadata_table.iterrows():
    if row["domain"] != "hatecrime":
        continue

    df = row["data"]        # The actual LLM-generated dataset
    model = row["model"]
    run   = row["run"]
    shot  = row["shot"]

    overall_BR, per_attr_BR, per_attr_BR_probs = base_rate_multiclass(
        df,
        sensitive_attrs=["offender_ethnicity_Mapped", "offender_race_Mapped"],
        outcome_col="is_violent",
        positive_label="non-violent"
    )

    results.append({
        "model": model,
        "run": run,
        "shot": shot,
        "br_overall": overall_BR,
        "br_race": per_attr_BR.get("offender_race_Mapped"),
        "br_ethnicity": per_attr_BR.get("offender_ethnicity_Mapped"),
    })

hatecrime_BR_results = pd.DataFrame(results)
#print(hatecrime_BR_results)

#Save the results in a csv file
output_path = PROJECT_ROOT / "analysis" / "bias_rq_02" / "hatecrime_e03_base_rate.csv"
hatecrime_BR_results.to_csv(output_path, index=False)
print("Saved real metrics to:", output_path)

Saved real metrics to: /Users/estref/Desktop/master_thesis/analysis/bias_rq_02/hatecrime_e03_base_rate.csv


In [35]:

HATECRIME_SENSITIVE = ["offender_ethnicity", "offender_race"] # for llm generated datasets the SA have _Mapped suffix
HATECRIME_OUTCOME_COL = "is_violent"          # contains "violent" / "non-violent"
HATECRIME_POSITIVE_LABEL = "non-violent"      # Y = 1

# REAL DATASET BARE RATE PARITY
overall_brp, per_attr_brp, per_attr_brp_probs = base_rate_parity(
    hatecrime_real, 
    HATECRIME_SENSITIVE, 
    HATECRIME_OUTCOME_COL, 
    HATECRIME_POSITIVE_LABEL
)

#REAL DATASET DISPARATE IMPACT
overall_di, per_attr_di, per_attr_di_probs = disparate_impact_multiclass(
    hatecrime_real,
    HATECRIME_SENSITIVE,
    HATECRIME_OUTCOME_COL,
    HATECRIME_POSITIVE_LABEL
)

#REAL DATASET BASE RATE
overall_br, per_attr_br, per_attr_br_probs = base_rate_multiclass(
    hatecrime_real,
    HATECRIME_SENSITIVE,
    HATECRIME_OUTCOME_COL,
    HATECRIME_POSITIVE_LABEL
)


# Put into a DataFrame
real_metrics_hatecrime = pd.DataFrame([{
    "overall_brp": overall_brp,
    "overall_di": overall_di,
    "overall_br": overall_br
}])

# Save to same directory you used earlier
output_path = PROJECT_ROOT / "analysis" / "bias_rq_02" / "hatecrime_real_metrics.csv"
real_metrics_hatecrime.to_csv(output_path, index=False)
print("Saved real metrics to:", output_path)



Saved real metrics to: /Users/estref/Desktop/master_thesis/analysis/bias_rq_02/hatecrime_real_metrics.csv


<b style="color:yellow;"> Calculating the metrics for <b style="color:red;"> LENDING</b>

<b style="color:yellow;"> 1. Base Rate Parity </b>

In [37]:
LENDING_SENSITIVE = ["derived_race", "derived_sex", "derived_ethnicity"]
LENDING_OUTCOME_COL = "loan_approved"          # contains "1" / "0"
LENDING_POSITIVE_LABEL = 1.0      # Y = 1

# Real dataset
# overall_bp_LENDING, per_attr_bp_LENDING, per_attr_probs_LENDING = base_rate_parity(
#     lending_real, 
#     LENDING_SENSITIVE, 
#     LENDING_OUTCOME_COL, 
#     LENDING_POSITIVE_LABEL
# )

# print("Overall BP:", overall_bp_LENDING)
#print("Per-attribute BP:", per_attr_bp)

results = []

for idx, row in metadata_table.iterrows():
    if row["domain"] != "lending":
        continue

    df = row["data"]        # The actual LLM-generated dataset
    model = row["model"]
    run   = row["run"]
    shot  = row["shot"]

    overall_bp, per_attr_bp, per_attr_probs = base_rate_parity(
        df,
        sensitive_attrs=["derived_race_Mapped", "derived_sex_Mapped", "derived_ethnicity_Mapped"],
        outcome_col="loan_approved",
        positive_label=1.0
    )

    results.append({
        "model": model,
        "run": run,
        "shot": shot,
        "bp_overall": overall_bp,
        "bp_race": per_attr_bp.get("derived_race_Mapped"),
        "bp_ethnicity": per_attr_bp.get("derived_ethnicity_Mapped"),
        "bp_sex": per_attr_bp.get("derived_sex_Mapped"),
    })

hatecrime_BRP_results = pd.DataFrame(results)
#print(hatecrime_BRP_results)

#Saving the results in a csv file
output_path = PROJECT_ROOT / "analysis" / "bias_rq_02" / "lending_e01_base_rate_parity.csv"
hatecrime_BRP_results.to_csv(output_path, index=False)
print("Saved real metrics to:", output_path)



                           model   run  shot  bp_overall   bp_race  \
0          kimi-k2-instruct-0905  run2   few    0.477447  0.542373   
1          kimi-k2-instruct-0905  run2   one    0.000000       NaN   
2          kimi-k2-instruct-0905  run2  zero    0.011683  0.021739   
3          kimi-k2-instruct-0905  run3   few    0.186835  0.222222   
4          kimi-k2-instruct-0905  run3   one         NaN       NaN   
5          kimi-k2-instruct-0905  run3  zero    0.014908  0.025126   
6          llama-3.1-8b-instruct  run3   few    0.289060  0.611111   
7          llama-3.1-8b-instruct  run3   one    0.088535  0.207547   
8          llama-3.1-8b-instruct  run3  zero    0.326517  0.127160   
9   qwen3-coder-30b-a3b-instruct  run1   few    0.212893  0.303030   
10  qwen3-coder-30b-a3b-instruct  run1   one    0.287722  0.480000   
11  qwen3-coder-30b-a3b-instruct  run1  zero    0.063020  0.114286   
12         llama-3.1-8b-instruct  run2   few    0.125611  0.139831   
13         llama-3.1

<b style="color:yellow;"> 2. Disparate Impact</b>

In [39]:
results = []

for idx, row in metadata_table.iterrows():
    if row["domain"] != "lending":
        continue

    df = row["data"]        # The actual LLM-generated dataset
    model = row["model"]
    run   = row["run"]
    shot  = row["shot"]

    overall_di, per_attr_di, per_attr_di_probs = disparate_impact_multiclass(
        df,
        sensitive_attrs=["derived_race_Mapped", "derived_sex_Mapped", "derived_ethnicity_Mapped"],
        outcome_col="loan_approved",
        positive_label=1.0
    )

    results.append({
        "model": model,
        "run": run,
        "shot": shot,
        "di_overall": overall_di,
        "di_race": per_attr_di.get("derived_race_Mapped"),
        "di_ethnicity": per_attr_di.get("derived_ethnicity_Mapped"),
        "di_sex": per_attr_di.get("derived_sex_Mapped"),
    })

lending_di_results = pd.DataFrame(results)
#print(lending_di_results)

#Saving the results in a csv file
output_path = PROJECT_ROOT / "analysis" / "bias_rq_02" / "lending_e02_disparate_impact.csv"
lending_di_results.to_csv(output_path, index=False)
print("Saved real metrics to:", output_path)


Saved real metrics to: /Users/estref/Desktop/master_thesis/analysis/bias_rq_02/lending_e02_disparate_impact.csv


<b style="color:yellow;"> 3. BASE RATE </b>

In [44]:
results = []

for idx, row in metadata_table.iterrows():
    if row["domain"] != "lending":
        continue

    df = row["data"]        # The actual LLM-generated dataset
    model = row["model"]
    run   = row["run"]
    shot  = row["shot"]

    overall_di, per_attr_di, per_attr_di_probs = base_rate_multiclass(
        df,
        sensitive_attrs=["derived_race", "derived_sex", "derived_ethnicity"],
        outcome_col="loan_approved",
        positive_label=1.0
    )

    results.append({
        "model": model,
        "run": run,
        "shot": shot,
        "br_overall": overall_di,
        "br_race": per_attr_di.get("derived_race"),
        "br_ethnicity": per_attr_di.get("derived_ethnicity"),
        "br_sex": per_attr_di.get("derived_sex"),
    })

lending_br_results = pd.DataFrame(results)
#print(lending_br_results)


#Saving the results in a csv file
output_path = PROJECT_ROOT / "analysis" / "bias_rq_02" / "lending_e03_base_rate.csv"
lending_br_results.to_csv(output_path, index=False)
print("Saved real metrics to:", output_path)



Saved real metrics to: /Users/estref/Desktop/master_thesis/analysis/bias_rq_02/lending_e03_base_rate.csv


In [42]:
# Saving the metrics for the real dataset

LENDING_SENSITIVE = ["derived_race", "derived_sex", "derived_ethnicity"]
LENDING_OUTCOME_COL = "loan_approved"          # contains "1" / "0"
LENDING_POSITIVE_LABEL = 1.0      # Y = 1

# REAL DATASET BARE RATE PARITY
overall_brp_LENDING, per_attr_brp_LENDING, per_attr_brp_probs_LENDING = base_rate_parity(
    lending_real, 
    LENDING_SENSITIVE, 
    LENDING_OUTCOME_COL, 
    LENDING_POSITIVE_LABEL
)

#REAL DATASET DISPARATE IMPACT
overall_di_LENDING, per_attr_di_LENDING, per_attr_di_probs_LENDING = disparate_impact_multiclass(
    lending_real,
    LENDING_SENSITIVE,
    LENDING_OUTCOME_COL,
    LENDING_POSITIVE_LABEL
)

#REAL DATASET BASE RATE
overall_br_LENDING, per_attr_br_LENDING, per_attr_br_probs_LENDING = base_rate_multiclass(
    lending_real,
    LENDING_SENSITIVE,
    LENDING_OUTCOME_COL,
    LENDING_POSITIVE_LABEL
)


# Put into a DataFrame
real_metrics_lending = pd.DataFrame([{
    "overall_brp": overall_brp_LENDING,
    "overall_di": overall_di_LENDING,
    "overall_br": overall_br_LENDING
}])

# Save to same directory you used earlier
output_path = PROJECT_ROOT / "analysis" / "bias_rq_02" / "lending_real_metrics.csv"
real_metrics_lending.to_csv(output_path, index=False)
print("Saved real metrics to:", output_path)



Saved real metrics to: /Users/estref/Desktop/master_thesis/analysis/bias_rq_02/lending_real_metrics.csv
